# Individual Exercise: Mini Data Quality Audit

Homework notebook for Week 2.


## Instructions

Perform a mini data quality audit.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

data_path = Path(r"D:/Ritu/Data_Management/week2_customer_transactions_messy.csv")

df = pd.read_csv(data_path)
print('Loaded:', data_path)
print('Shape:', df.shape)
df.head()


Loaded: D:\Ritu\Data_Management\week2_customer_transactions_messy.csv
Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


## Required tasks

1. describe the dataset briefly
2. identify issues by dimension
3. compute at least 3 KPIs
4. define at least 3 validation rules
5. create a short audit summary
6. recommend cleaning steps for next chapter (do not implement full pipelines)


## Task 1 - Dataset description

### Your answer

The dataset contains **11 customer transaction records** across **9 columns**, covering early 2026. Each row represents a single transaction with the following fields:

- **`transaction_id`** — transaction identifier (expected to be unique)
- **`customer_id`** — identifier of the customer who made the transaction
- **`transaction_date`** — the date on which the transaction occurred
- **`amount`** — monetary value of the transaction (numeric)
- **`currency`** — ISO currency code (expected: EUR, USD, etc.)
- **`payment_method`** — how the transaction was paid (card, bank_transfer, cash)
- **`status`** — lifecycle state of the transaction (completed, pending, cancelled)
- **`region`** — country/region code (expected: 2-letter ISO, uppercase)
- **`last_updated`** — date the record was last modified (used for timeliness checks)

This dataset is so messy and serves as the basis for a quality audit. A quick inspection reveals multiple data-quality issues across the six standard quality dimensions (completeness, uniqueness, validity, consistency, accuracy, timeliness), which will be diagnosed in the tasks below.

In [ ]:
#overview of the data
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nMissing per column:')
print(df.isna().sum())

Shape: (11, 9)

Dtypes:
transaction_id       object
customer_id          object
transaction_date     object
amount              float64
currency             object
payment_method       object
status               object
region               object
last_updated         object
dtype: object

Missing per column:
transaction_id      0
customer_id         1
transaction_date    0
amount              1
currency            0
payment_method      1
status              0
region              0
last_updated        1
dtype: int64


## Task 2 - Issues by dimension


In [5]:
issue_table = pd.DataFrame(
    [
        ['Missing customer_id (row T0004)','Completeness','Transaction cannot be attributed to a customer; breaks customer-level analytics'],
        ['Missing amount (row T0009)','Completeness','Revenue totals and averages are understated'],
        ['Missing payment_method (row T0010)','Completeness','Payment-mix reporting is incomplete'],
        ['Missing last_updated (row T0009)','Completeness','Record freshness cannot be assessed'],
        ['Duplicate transaction_id T0006','Uniqueness','Double counting of revenue and transaction volume'],
        ['Invalid calendar date (2026-02-30, row T0007)','Validity','Parses as NaT; row dropped by date-based joins and time-series aggregations'],
        ['Mixed date formats (2026/01/06, 06-01-2026)','Validity','Parsing fragile; requires format="mixed" or per-row format detection'],
        ['Negative amount (-35.00, row T0003)','Validity','Violates business rule that transactions are non-negative'],
        ['Currency inconsistency (EUR vs EURO)','Consistency','Grouping and FX conversion break if codes are not standardised'],
        ['Payment method casing (card vs CARD)','Consistency','Category counts are split across duplicate labels'],
        ['Region casing (DE vs de)','Consistency','Region-level aggregations under-count Germany'],
        ['Amount outlier (1,000,000 EUR, row T0008)','Accuracy','Dominates revenue totals and averages; likely a data-entry error'],
        ['Amount = 0 (row T0002)','Accuracy','Likely placeholder/default; distorts mean ticket size'],
        ['last_updated before transaction_date (row T0003)','Timeliness','Logically impossible; indicates wrong update or transaction date'],
        ['last_updated 3+ months after transaction (T0006, T0007)','Timeliness','Stale records; updates lag far behind the event'],
    ],
    columns=['Issue', 'Dimension', 'Impact']
)
issue_table

,Issue,Dimension,Impact
0,Missing customer_id (row T0004),Completeness,Transaction cannot be attributed to a customer...
1,Missing amount (row T0009),Completeness,Revenue totals and averages are understated
2,Missing payment_method (row T0010),Completeness,Payment-mix reporting is incomplete
3,Missing last_updated (row T0009),Completeness,Record freshness cannot be assessed
4,Duplicate transaction_id T0006,Uniqueness,Double counting of revenue and transaction volume
5,"Invalid calendar date (2026-02-30, row T0007)",Validity,Parses as NaT; row dropped by date-based joins...
6,"Mixed date formats (2026/01/06, 06-01-2026)",Validity,"Parsing fragile; requires format=""mixed"" or pe..."
7,"Negative amount (-35.00, row T0003)",Validity,Violates business rule that transactions are n...
8,Currency inconsistency (EUR vs EURO),Consistency,Grouping and FX conversion break if codes are ...
9,Payment method casing (card vs CARD),Consistency,Category counts are split across duplicate labels


## Task 3 - KPI calculations


In [6]:
kpis = {}

# 1. Completeness Rate: share of non-null cells across the whole table
kpis['Completeness Rate'] = 1 - (df.isna().sum().sum() / (df.shape[0] * df.shape[1]))

# 2. Duplication Rate: share of rows whose transaction_id is a duplicate
kpis['Duplication Rate'] = df.duplicated(subset=['transaction_id']).mean()

# 3. Error Rate: rows failing at least one validity check
#    (amount missing, amount negative, or invalid date)
amount = pd.to_numeric(df['amount'], errors='coerce')
date_ok = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').notna()
kpis['Error Rate'] = (amount.isna() | (amount < 0) | ~date_ok).mean()

# 4. Uniqueness Rate: inverse of duplication, expressed positively
kpis['Uniqueness Rate'] = 1 - df.duplicated(subset=['transaction_id']).mean()

# 5. Currency Consistency: share of rows with a valid ISO-4217 code
kpis['Currency Consistency'] = df['currency'].str.upper().isin(['EUR', 'USD', 'GBP']).mean()

pd.DataFrame({'KPI': list(kpis.keys()), 'Value': [round(v, 4) for v in kpis.values()]})

,KPI,Value
0,Completeness Rate,0.9596
1,Duplication Rate,0.0909
2,Error Rate,0.2727
3,Uniqueness Rate,0.9091
4,Currency Consistency,0.9091


### My KPI interpretation

- **Completeness Rate ≈ 95.96 %** — Overall cell-level completeness looks high, but this single number hides the fact that four different columns each have one missing value, and some of them (`customer_id`, `amount`) are critical.
- **Duplication Rate ≈ 9.09 %** — Roughly 1 in 11 rows is a duplicate transaction_id (T0006). For a column that should be a primary key, *any* duplication is a hard failure.
- **Error Rate ≈ 27.27 %** — More than a quarter of rows fail at least one validity rule (missing amount, negative amount, or unparseable date). This is the strongest warning signal in the audit.
- **Uniqueness Rate ≈ 90.91 %** — Mirror of the duplication rate; confirms the `transaction_id` primary-key assumption is broken.
- **Currency Consistency ≈ 90.91 %** — One row uses "EURO" instead of "EUR". Low-volume but it breaks any `GROUP BY currency`.

**Takeaway:** cell-level completeness is misleadingly reassuring. The Error Rate and Duplication Rate are the KPIs that actually reflect how unsafe this data is to use as-is.

## Task 4 - Validation rules

Six validation rules covering the most important dimensions. Each rule returns a boolean mask of violating rows (`True` = the row violates the rule).

In [7]:
rules = {
    # Completeness
    'transaction_id_required': df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip() == ''),
    'customer_id_required': df['customer_id'].isna(),
    # Validity
    'amount_non_negative': pd.to_numeric(df['amount'], errors='coerce') < 0,
    'transaction_date_valid': pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna(),
    # Uniqueness
    'no_duplicate_transactions': df.duplicated(subset=['transaction_id'], keep=False),
    # Consistency
    'currency_in_iso_list': ~df['currency'].str.upper().isin(['EUR', 'USD', 'GBP']),
}
pd.DataFrame({k: int(v.sum()) for k, v in rules.items()}, index=['affected_rows']).T

,affected_rows
transaction_id_required,0
customer_id_required,1
amount_non_negative,1
transaction_date_valid,1
no_duplicate_transactions,2
currency_in_iso_list,1


## Task 5 - Audit summary


In [8]:
audit_summary = pd.DataFrame(
    [
        ['Missing customer_id', 1, 'High', 'Impute only if recoverable from upstream source; otherwise flag & quarantine'],
        ['Missing amount', 1, 'High', 'Quarantine row; do not impute monetary values'],
        ['Missing payment_method', 1, 'Medium', 'Set to "unknown" category to preserve the row for revenue totals'],
        ['Missing last_updated', 1, 'Low', 'Default to transaction_date as best-effort proxy'],
        ['Duplicate transaction_id', 2, 'High', 'Deduplicate on transaction_id keeping the first (or latest last_updated) occurrence'],
        ['Invalid calendar date', 1, 'High', 'Quarantine row; request corrected value from source system'],
        ['Mixed date formats', 11, 'Medium', 'Standardise to ISO-8601 (YYYY-MM-DD) using pd.to_datetime(format="mixed")'],
        ['Negative amount', 1, 'High', 'Quarantine or move to a separate refunds table depending on business rule'],
        ['Currency inconsistency', 1, 'Medium', 'Map synonyms ("EURO" -> "EUR") to canonical ISO-4217 codes'],
        ['Payment method casing', 2, 'Low', 'Normalise to lowercase (.str.lower()) before aggregation'],
        ['Region casing', 1, 'Low', 'Normalise to uppercase 2-letter ISO codes'],
        ['Amount outlier (1,000,000)', 1, 'Medium', 'Flag for manual review; confirm with source whether it is legitimate'],
        ['Amount = 0', 1, 'Medium', 'Flag for review; may indicate cancelled or placeholder transaction'],
        ['last_updated before tx_date', 1, 'Medium', 'Flag logical inconsistency; escalate to source system owner'],
    ],
    columns=['issue_type', 'affected_rows', 'severity', 'recommended_next_action']
)
audit_summary


,issue_type,affected_rows,severity,recommended_next_action
0,Missing customer_id,1,High,Impute only if recoverable from upstream sourc...
1,Missing amount,1,High,Quarantine row; do not impute monetary values
2,Missing payment_method,1,Medium,"Set to ""unknown"" category to preserve the row ..."
3,Missing last_updated,1,Low,Default to transaction_date as best-effort proxy
4,Duplicate transaction_id,2,High,Deduplicate on transaction_id keeping the firs...
5,Invalid calendar date,1,High,Quarantine row; request corrected value from s...
6,Mixed date formats,11,Medium,Standardise to ISO-8601 (YYYY-MM-DD) using pd....
7,Negative amount,1,High,Quarantine or move to a separate refunds table...
8,Currency inconsistency,1,Medium,"Map synonyms (""EURO"" -> ""EUR"") to canonical IS..."
9,Payment method casing,2,Low,Normalise to lowercase (.str.lower()) before a...


## Task 6 - Recommended cleaning steps for next chapter

These are recommendations only — not implemented here.

1. **Remove duplicates** — drop duplicate `transaction_id` rows (keep the first one). Example: `df.drop_duplicates(subset=['transaction_id'], keep='first')`.

2. **Fix date formats** — convert all dates to a standard format using `pd.to_datetime(..., format='mixed', errors='coerce')`. Invalid dates like `2026-02-30` will become `NaT` and can be moved to a separate table for review.

3. **Standardise text columns** — lowercase `payment_method` (so "card" and "CARD" match), uppercase `region` (so "DE" and "de" match), and replace "EURO" with "EUR".

4. **Handle missing values differently per column** — drop rows missing `customer_id` or `amount` (critical fields), fill missing `payment_method` with "unknown", and use `transaction_date` as a fallback for missing `last_updated`.

5. **Flag outliers instead of deleting them** — add a new column like `is_outlier = True` for amounts that look wrong (e.g. 1,000,000 EUR or 0). This keeps the row but lets analysts exclude it easily.

6. **Add business rule checks** — reject rows where `amount < 0` (unless it's a refund), and reject rows where `last_updated` is earlier than `transaction_date`.

## Reflection questions

**1. Which KPI gave the strongest signal?**

The **Error Rate (≈ 27%)** gave the strongest signal. The Completeness Rate looked fine at ~96%, which was misleading. But the Error Rate showed that more than 1 in 4 rows fail a basic validity check (missing amount, negative amount, or invalid date). This is the KPI I would report first to a stakeholder, because it directly tells you how much of the data is unsafe to use.

**2. Which issue should be escalated first?**

The **duplicate `transaction_id` (T0006)** should be escalated first. `transaction_id` is supposed to be a unique key, so duplicates silently double-count revenue in every report. It's also easy to fix in the pipeline but hard to detect once dashboards are already live. A close second is the **1,000,000 EUR outlier**, because one wrong row of that size can distort totals and averages across the whole dataset.

**3. What additional metadata would improve this audit?**

- **Source system name** — to know which team to contact for fixes.
- **Ingestion timestamp** — separate from `last_updated`, to tell pipeline delays apart from source delays.
- **Column-level schema** with expected data types, value ranges, and whether nulls are allowed — so validation rules can be generated automatically.
- **Business rule catalogue** (e.g. "amount must be ≥ 0 unless status = refund") — so validity checks are based on documented rules, not guesses.
- **Historical KPI values** — one snapshot tells you the current state; a time series tells you whether quality is improving or getting worse.

In [10]:
# Evidence for Q1: 
print("Completeness Rate:", round(kpis['Completeness Rate'], 4), "-> looks healthy at ~96%")
print("Error Rate:       ", round(kpis['Error Rate'], 4), "-> but 27% of rows fail validity checks")

Completeness Rate: 0.9596 -> looks healthy at ~96%
Error Rate:        0.2727 -> but 27% of rows fail validity checks


In [11]:
# Evidence for Q2: show the duplicate transaction_id rows
df[df.duplicated(subset=['transaction_id'], keep=False)]

,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15


In [13]:
# Evidence for Q3: logical inconsistency proves we need better metadata/rules
td = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
lu = pd.to_datetime(df['last_updated'], errors='coerce', format='mixed')
df[lu < td][['transaction_id', 'transaction_date', 'last_updated']]

,transaction_id,transaction_date,last_updated
2,T0003,06-01-2026,2026-01-07


## Bonus (recommended): write your own audit function

Create at least one helper function with clear parameters and docstring.
This demonstrates that your logic is reusable and understandable.


In [14]:
def summarize_rule_violations(rule_dictionary, total_rows=None, severity_map=None):
    """Summarize affected-row counts for each validation rule.

    Parameters
    ----------
    rule_dictionary : dict[str, pd.Series]
        Dictionary where keys are rule names and values are boolean masks
        (True = the row violates the rule).
    total_rows : int, optional
        Total number of rows in the dataset. If provided, a violation_rate
        column is added. Defaults to None.
    severity_map : dict[str, str], optional
        Optional mapping from rule name to severity label (High/Medium/Low).
        Missing entries are filled with 'Unspecified'.

    Returns
    -------
    pd.DataFrame
        One row per rule, sorted by affected_rows descending.
    """
    rows = []
    for name, mask in rule_dictionary.items():
        entry = {'rule_name': name, 'affected_rows': int(mask.sum())}
        if total_rows:
            entry['violation_rate'] = round(int(mask.sum()) / total_rows, 4)
        if severity_map is not None:
            entry['severity'] = severity_map.get(name, 'Unspecified')
        rows.append(entry)
    return pd.DataFrame(rows).sort_values('affected_rows', ascending=False).reset_index(drop=True)


# Example usage with the rules dictionary defined earlier
severity_map = {
    'transaction_id_required': 'High',
    'customer_id_required': 'High',
    'amount_non_negative': 'High',
    'transaction_date_valid': 'High',
    'no_duplicate_transactions': 'High',
    'currency_in_iso_list': 'Medium',
}

summarize_rule_violations(rules, total_rows=len(df), severity_map=severity_map)

,rule_name,affected_rows,violation_rate,severity
0,no_duplicate_transactions,2,0.1818,High
1,customer_id_required,1,0.0909,High
2,transaction_date_valid,1,0.0909,High
3,amount_non_negative,1,0.0909,High
4,currency_in_iso_list,1,0.0909,Medium
5,transaction_id_required,0,0.0000,High


### Explain your function parameters

- **`rule_dictionary`** is the required input. It's a dictionary where each key is a rule name (like `'amount_non_negative'`) and each value is a boolean pandas Series marking which rows violate that rule. It has no default value because the function has nothing to summarise without it — this is the core data the function operates on. Passing a different set of rules changes which issues appear in the output table, so this parameter directly controls what the audit covers.

- **`total_rows`** defaults to `None` so the function works in two modes: raw counts only, or counts plus rates. When I pass `len(df)`, the function adds a `violation_rate` column calculated as `affected_rows / total_rows`. I chose `None` as the default because not every use case needs rates (e.g. quick debugging), and forcing the user to always provide it would make the function less flexible. Without it, a rule affecting 2 out of 11 rows looks the same as 2 out of 1,000,000 — which hides how serious the issue really is.

- **`severity_map`** also defaults to `None` to keep the function lightweight when you only need counts. When provided, it adds a `severity` column (High/Medium/Low) so the audit table can be prioritised, not just sorted by count. I used `.get(name, 'Unspecified')` so missing entries don't crash the function — they just get labelled as unspecified. Changing the severity mapping changes how issues are ranked for escalation: a rule with few affected rows but High severity (like a duplicate primary key) should be fixed before a Low-severity rule with many affected rows (like casing inconsistencies).

## Submission checklist

- [x] Dataset described
- [x] Issues mapped to dimensions
- [x] At least 3 KPIs calculated (5 provided)
- [x] At least 3 validation rules defined (6 provided)
- [x] Audit summary completed
- [x] Cleaning recommendations proposed (not implemented)